# Tarang Expense Receipt Reader — Accuracy & Policy Evaluation

This notebook benchmarks field extraction accuracy and policy compliance decisions against the 283 ground-truth claims in `claims_train.csv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import settings
from src.pipeline import ExpensePipeline
from src.extractor import normalize_date_to_iso
from src.rules import normalize_amount

print("Pipeline loaded successfully!")

## 1. Load Ground Truth Training Dataset

In [ ]:
train_csv = settings.resolve_raw_file("claims_train.csv")
train_df = pd.read_csv(train_csv)
print(f"Total training claims: {len(train_df)}")
print(train_df["status"].value_counts())
train_df.head(3)

## 2. Run Pipeline Extraction on Training Claims

In [ ]:
pipeline = ExpensePipeline()
seen_receipts = set()
predictions = []

for idx, row in train_df.iterrows():
    res = pipeline.process_claim(row.to_dict(), seen_receipts=seen_receipts)
    predictions.append(res)

pred_df = pd.DataFrame(predictions)
print(f"Processed {len(pred_df)} claims successfully!")
pred_df.head(3)

## 3. Field-by-Field Accuracy Evaluation

In [ ]:
def clean_text(s):
    if s is None or pd.isna(s):
        return ""
    return "".join(c for c in str(s).lower() if c.isalnum())

total = len(train_df)
v_matches = sum(1 for i, r in train_df.iterrows() if clean_text(pred_df.loc[i, "vendor"]) in clean_text(r["clerk_vendor"]) or clean_text(r["clerk_vendor"]) in clean_text(pred_df.loc[i, "vendor"]) if clean_text(r["clerk_vendor"])) 
d_matches = sum(1 for i, r in train_df.iterrows() if pred_df.loc[i, "receipt_date"] == normalize_date_to_iso(str(r["clerk_bill_date"])))
b_matches = sum(1 for i, r in train_df.iterrows() if clean_text(pred_df.loc[i, "receipt_number"]) == clean_text(r["clerk_bill_no"]) or (not clean_text(r["clerk_bill_no"]) and not clean_text(pred_df.loc[i, "receipt_number"])))
a_matches = sum(1 for i, r in train_df.iterrows() if pd.notna(r["clerk_bill_amount"]) and pred_df.loc[i, "total_amount"] is not None and abs(float(r["clerk_bill_amount"]) - float(pred_df.loc[i, "total_amount"])) <= 1.0)
g_matches = sum(1 for i, r in train_df.iterrows() if clean_text(pred_df.loc[i, "gstin"]) == clean_text(r["clerk_gstin"]))

metrics = {
    "Field": ["Vendor Name", "Receipt Date", "Bill Number", "Total Amount", "GSTIN", "OVERALL AVERAGE"],
    "Matches": [v_matches, d_matches, b_matches, a_matches, g_matches, "-"],
    "Accuracy (%)": [
        round(v_matches / total * 100, 2),
        round(d_matches / total * 100, 2),
        round(b_matches / total * 100, 2),
        round(a_matches / total * 100, 2),
        round(g_matches / total * 100, 2),
        round((v_matches + d_matches + b_matches + a_matches + g_matches) / (5 * total) * 100, 2)
    ]
}
pd.DataFrame(metrics)

## 4. Policy Decision Distribution & Risk Flags

In [ ]:
print("Policy Decisions Breakdown:")
print(pred_df["decision"].value_counts())

print("\nSample Reasons Flagged for Review:")
pred_df[pred_df["decision"] == "review"][["claim_id", "total_amount", "reason"]].head(10)